# Feature Preprocessing & Engine-Wise Data Splitting
### B.Tech Mini Project: Aircraft Engine Predictive Maintenance (NASA C-MAPSS FD001)

This notebook implements the core data preparation and feature scaling pipeline for the **NASA C-MAPSS FD001** dataset.

#### Pipeline Stages:
1. **Stage 1 - Load & Prepare Data:** Ingest training, test, and test RUL sets; generate capped training RUL labels ($RUL_{max} = 125$).
2. **Stage 2 - Engine-Wise Splitting:** Partition the 100 training engines into 80% train (80 engines) and 20% validation (20 engines) by entire engine trajectory to prevent data leakage.
3. **Stage 3 - Feature Selection:** Dynamically evaluate variance on the training partition only; filter uninformative zero-variance features while retaining `sensor_6`.
4. **Stage 4 - Feature Scaling:** Fit `StandardScaler` strictly on the training partition; transform validation and test sets without data leakage.
5. **Stage 5 - Strict Pipeline Validation:** Execute assertions verifying disjoint splits, correct shapes, zero mean / unit variance, and absence of nulls/infinities.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("NumPy version  :", np.__version__)
print("Pandas version :", pd.__version__)
print("Libraries successfully imported.")

NumPy version  : 2.5.4
Pandas version : 3.1.0
Libraries successfully imported.


---
## Stage 1: Load and Prepare Data

We reuse the standardized 26-column naming schema:
- **Identifiers:** `unit` (engine ID), `cycle` (operational flight cycle)
- **Operational Settings:** `op_setting_1`, `op_setting_2`, `op_setting_3`
- **Telemetry Channels:** 21 sensor measurements (`sensor_1` through `sensor_21`)

Training target labels are generated using the established piecewise constant-linear formula:
$$\text{raw\_RUL} = \max(\text{cycle}) - \text{cycle}, \quad \text{RUL} = \min(125, \text{raw\_RUL})$$

In [2]:
# 1. Define standardized column schema
columns = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

# 2. Resolve dataset directory path
DATA_DIR = Path("../data/raw/CMaps") if Path("../data/raw/CMaps").exists() else Path("data/raw/CMaps")

# 3. Ingest raw datasets
train_df = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv(DATA_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_df = pd.read_csv(DATA_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

# 4. Generate capped training RUL labels
train_df["max_cycle"] = train_df.groupby("unit")["cycle"].transform("max")
train_df["raw_RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df["RUL"] = np.minimum(125, train_df["raw_RUL"])

# 5. Verify shapes and data integrity
print("=" * 65)
print("STAGE 1: DATA INGESTION & INTEGRITY CHECK")
print("=" * 65)
print(f"Training Data (train_df) : {train_df.shape} (Missing values: {train_df.isnull().sum().sum()})")
print(f"Test Data (test_df)      : {test_df.shape} (Missing values: {test_df.isnull().sum().sum()})")
print(f"Ground Truth RUL (rul_df): {rul_df.shape} (Missing values: {rul_df.isnull().sum().sum()})")
print(f"Unique Training Engines  : {train_df['unit'].nunique()}")
print(f"Unique Test Engines      : {test_df['unit'].nunique()}")
print("=" * 65)

STAGE 1: DATA INGESTION & INTEGRITY CHECK
Training Data (train_df) : (20631, 29) (Missing values: 0)
Test Data (test_df)      : (13096, 26) (Missing values: 0)
Ground Truth RUL (rul_df): (100, 1) (Missing values: 0)
Unique Training Engines  : 100
Unique Test Engines      : 100


---
## Stage 2: Engine-Wise Training / Validation Split

### Why Engine-Wise Splitting is Mandatory
In run-to-failure prognostics, observations from the same engine are sequentially and auto-correlatively dependent.
- If a conventional **random row-wise split** were applied, cycles from the same engine would appear simultaneously in both training and validation sets. The model would effectively memorize engine-specific degradation signatures, leading to **severe data leakage** and artificially inflated validation metrics.
- To maintain statistical independence and mimic real-world deployment, we perform an **engine-wise split**: unique engine units are split into disjoint sets ($80\%$ train, $20\%$ validation), preserving complete engine trajectories.

In [3]:
# 1. Extract unique engine IDs from training dataset
engine_ids = train_df["unit"].unique()

# 2. Split engine IDs (80% train, 20% validation, fixed seed = 42)
train_units, val_units = train_test_split(
    engine_ids, test_size=0.20, random_state=42
)

# 3. Filter complete engine trajectories for each split
train_split_df = train_df[train_df["unit"].isin(train_units)].copy().reset_index(drop=True)
val_split_df = train_df[train_df["unit"].isin(val_units)].copy().reset_index(drop=True)

# 4. Verify partition disjointness and summary metrics
overlap = set(train_units).intersection(set(val_units))
assert len(overlap) == 0, f"Data Leakage Error: Overlapping engines found: {overlap}"

print("=" * 65)
print("STAGE 2: ENGINE-WISE SPLIT SUMMARY")
print("=" * 65)
print(f"Training Partition   : {len(train_units)} engines ({len(train_split_df):,} rows, {len(train_split_df)/len(train_df)*100:.2f}% of data)")
print(f"Validation Partition : {len(val_units)} engines ({len(val_split_df):,} rows, {len(val_split_df)/len(train_df)*100:.2f}% of data)")
print(f"Engine Overlap       : {len(overlap)} (Zero overlap confirmed)")
print("=" * 65)

STAGE 2: ENGINE-WISE SPLIT SUMMARY
Training Partition   : 80 engines (16,561 rows, 80.27% of data)
Validation Partition : 20 engines (4,070 rows, 19.73% of data)
Engine Overlap       : 0 (Zero overlap confirmed)


---
## Stage 3: Feature Selection

### Candidate Feature Pool
The candidate input features consist of all 3 operational settings (`op_setting_1, 2, 3`) and 21 sensors (`sensor_1` through `sensor_21`) — total **24 candidate features**.

### Zero-Variance Filtering Rule
- Variance is calculated strictly on `train_split_df` to prevent data leakage.
- Any feature with strictly zero variance (or $\text{nunique} = 1$) is completely uninformative and cannot be scaled or used by neural networks.
- Identifiers and target metadata (`unit`, `cycle`, `max_cycle`, `raw_RUL`, `RUL`) are excluded from model inputs.

### Treatment of Near-Constant `sensor_6`
- `sensor_6` (HPC inlet total pressure) exhibits two discrete states (21.60 and 21.61) with very low variance ($\approx 1.86 \times 10^{-6}$).
- As instructed, we **retain `sensor_6`** at this stage so subsequent sequence models or feature importance benchmarks can empirically evaluate whether its state transitions provide useful prognostic signal.

In [4]:
# 1. Define candidate feature columns (settings + sensors)
candidate_features = [f"op_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]

# 2. Compute variance, standard deviation, and unique counts strictly on train_split_df
feature_stats = pd.DataFrame({
    "variance": train_split_df[candidate_features].var(),
    "std": train_split_df[candidate_features].std(),
    "nunique": train_split_df[candidate_features].nunique()
})

# 3. Dynamically identify zero-variance features
zero_var_mask = (feature_stats["nunique"] == 1) | (feature_stats["variance"] < 1e-12)
zero_var_features = feature_stats[zero_var_mask].index.tolist()

# 4. Define selected feature set (retaining near-constant sensor_6)
selected_features = [f for f in candidate_features if f not in zero_var_features]

print("=" * 65)
print("STAGE 3: FEATURE VARIANCE & SELECTION SUMMARY")
print("=" * 65)
display(feature_stats.round(6))
print("-" * 65)
print(f"Removed Zero-Variance Features ({len(zero_var_features)}): {zero_var_features}")
print(f"\nNear-Constant Sensor Analysis (sensor_6):")
s6_counts = train_split_df['sensor_6'].value_counts(normalize=True).to_dict()
print(f"  • Variance : {feature_stats.loc['sensor_6', 'variance']:.6e}")
print(f"  • Distribution: {s6_counts}")
print(f"  • Decision : Retained for downstream modeling assessment.")
print(f"\nSelected Input Features ({len(selected_features)}): {selected_features}")
print("=" * 65)

STAGE 3: FEATURE VARIANCE & SELECTION SUMMARY


,variance,std,nunique
op_setting_1,0.000005,0.002191,156
op_setting_2,0.000000,0.000293,13
op_setting_3,0.000000,0.000000,1
sensor_1,0.000000,0.000000,1
sensor_2,0.248128,0.498124,305
sensor_3,37.032203,6.085409,2878
sensor_4,80.382265,8.965616,3893
sensor_5,0.000000,0.000000,1
sensor_6,0.000002,0.001364,2
sensor_7,0.770484,0.877772,501


-----------------------------------------------------------------
Removed Zero-Variance Features (7): ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']

Near-Constant Sensor Analysis (sensor_6):
  • Variance : 1.860184e-06
  • Distribution: {21.61: 0.9810397922830747, 21.6: 0.018960207716925307}
  • Decision : Retained for downstream modeling assessment.

Selected Input Features (17): ['op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


---
## Stage 4: Feature Scaling (StandardScaler)

### Standardization Protocol ($z = \frac{x - \mu}{\sigma}$)
- **Fit strictly on `train_split_df[selected_features]`**: Computes empirical feature means $\mu_{train}$ and standard deviations $\sigma_{train}$.
- **Transform `val_split_df` and `test_df` using the training-fitted scaler**: Ensures evaluation splits represent genuinely unseen observations.
- **Preserve Tracking Metadata**: `unit`, `cycle`, and `RUL` are preserved separately and concatenated with the scaled features into `train_scaled_df`, `val_scaled_df`, and `test_scaled_df`.

In [5]:
# 1. Initialize and fit StandardScaler strictly on training partition
scaler = StandardScaler()
train_scaled_features = scaler.fit_transform(train_split_df[selected_features])

# 2. Transform validation and test data using the fitted scaler
val_scaled_features = scaler.transform(val_split_df[selected_features])
test_scaled_features = scaler.transform(test_df[selected_features])

# 3. Construct clean structured DataFrames preserving identifiers and targets
train_scaled_df = pd.concat([
    train_split_df[["unit", "cycle"]],
    pd.DataFrame(train_scaled_features, columns=selected_features),
    train_split_df[["RUL"]]
], axis=1)

val_scaled_df = pd.concat([
    val_split_df[["unit", "cycle"]],
    pd.DataFrame(val_scaled_features, columns=selected_features),
    val_split_df[["RUL"]]
], axis=1)

test_scaled_df = pd.concat([
    test_df[["unit", "cycle"]],
    pd.DataFrame(test_scaled_features, columns=selected_features)
], axis=1)

print("=" * 65)
print("STAGE 4: SCALED DATAFRAMES OVERVIEW")
print("=" * 65)
print(f"train_scaled_df shape : {train_scaled_df.shape}  ({train_scaled_df['unit'].nunique()} engines, {len(train_scaled_df):,} rows)")
print(f"val_scaled_df shape   : {val_scaled_df.shape}    ({val_scaled_df['unit'].nunique()} engines, {len(val_scaled_df):,} rows)")
print(f"test_scaled_df shape  : {test_scaled_df.shape}   ({test_scaled_df['unit'].nunique()} engines, {len(test_scaled_df):,} rows)")
print("-" * 65)
print("First 5 rows of train_scaled_df:")
display(train_scaled_df.head())
print("=" * 65)

STAGE 4: SCALED DATAFRAMES OVERVIEW
train_scaled_df shape : (16561, 20)  (80 engines, 16,561 rows)
val_scaled_df shape   : (4070, 20)    (20 engines, 4,070 rows)
test_scaled_df shape  : (13096, 19)   (100 engines, 13,096 rows)
-----------------------------------------------------------------
First 5 rows of train_scaled_df:


,unit,cycle,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,2,1,-0.818512,2.043793,-1.581191,-1.092662,-1.963014,-7.193194,1.318090,-1.221017,-0.471297,-2.294365,1.242127,-0.500147,-0.315944,-1.373754,-1.426246,0.678684,1.564436,125
1,2,2,1.966124,-1.031739,-1.721722,-0.565155,-1.756664,0.139020,1.591518,-1.645675,-0.621789,-1.127747,1.745551,-1.615806,-0.654442,-0.675217,-0.779529,1.345844,1.098895,125
2,2,3,0.824880,1.018616,-2.263772,-0.356453,-1.106382,-7.193194,2.013053,-0.796358,-0.492481,-1.203012,1.582278,-1.476349,-0.169925,-1.648369,-1.426246,1.623827,1.252524,125
3,2,4,1.600926,-1.373465,-2.002785,-1.041719,-1.427619,0.139020,0.999091,-1.645675,-0.326102,-1.654606,1.459824,-2.313093,-0.177072,-1.072476,-1.426246,1.735021,1.975974,125
4,2,5,0.231433,1.360341,-1.902406,-1.883102,-0.709298,-7.193194,1.990267,-0.937911,-0.319923,-1.090114,1.160491,-2.173636,-0.369552,-1.475068,-2.072963,2.013004,1.237627,125


---
## Stage 5: Validation Checks & Assertions

We enforce six verification checks:
1. **Disjoint Engine Partitions:** No overlap between training and validation engine IDs.
2. **Engine Count Accuracy:** Exactly 80 training engines and 20 validation engines.
3. **Scaled Mean Centering:** Scaled training feature means are approximately zero ($\mu \approx 0$).
4. **Scaled Unit Variance:** Scaled training feature standard deviations are approximately one ($\sigma \approx 1$).
5. **Data Completeness:** Zero null, NaN, or infinite values in all transformed matrices.
6. **Scaler Consistency:** Validation and test splits transformed strictly using the training scaler's fitted attributes.

In [6]:
print("=" * 65)
print("STAGE 5: SYSTEMATIC PIPELINE VALIDATION")
print("=" * 65)

# Check 1: Engine ID disjointness
overlap = set(train_scaled_df["unit"]).intersection(set(val_scaled_df["unit"]))
assert len(overlap) == 0, f"Assertion 1 Failed: Overlapping engines: {overlap}"
print("✓ Check 1 Passed: Training and validation engine IDs do not overlap.")

# Check 2: Engine counts
n_train_units = train_scaled_df["unit"].nunique()
n_val_units = val_scaled_df["unit"].nunique()
assert n_train_units == 80, f"Assertion 2 Failed: Expected 80 train units, got {n_train_units}"
assert n_val_units == 20, f"Assertion 2 Failed: Expected 20 val units, got {n_val_units}"
print(f"✓ Check 2 Passed: Engine counts are correct (Train: {n_train_units}, Val: {n_val_units}).")

# Check 3: Scaled training feature means approx 0
train_means = train_scaled_df[selected_features].mean()
assert np.allclose(train_means, 0.0, atol=1e-3), "Assertion 3 Failed: Means not centered at 0!"
print(f"✓ Check 3 Passed: Training feature means are approx 0 (Max abs mean: {train_means.abs().max():.2e}).")

# Check 4: Scaled training feature standard deviations approx 1
train_stds = train_scaled_df[selected_features].std()
assert np.allclose(train_stds, 1.0, atol=1e-3), "Assertion 4 Failed: Standard deviations not approx 1!"
print(f"✓ Check 4 Passed: Training feature stds are approx 1 (Min: {train_stds.min():.4f}, Max: {train_stds.max():.4f}).")

# Check 5: No missing or infinite values
null_count = (
    train_scaled_df.isnull().sum().sum()
    + val_scaled_df.isnull().sum().sum()
    + test_scaled_df.isnull().sum().sum()
)
inf_count = (
    np.isinf(train_scaled_df[selected_features].values).sum()
    + np.isinf(val_scaled_df[selected_features].values).sum()
    + np.isinf(test_scaled_df[selected_features].values).sum()
)
assert null_count == 0, f"Assertion 5 Failed: Found {null_count} missing values!"
assert inf_count == 0, f"Assertion 5 Failed: Found {inf_count} infinite values!"
print("✓ Check 5 Passed: No missing or infinite values exist across all scaled matrices.")

# Check 6: Scaler fitted attributes consistency
assert hasattr(scaler, "mean_") and hasattr(scaler, "scale_"), "Assertion 6 Failed: Scaler not fitted!"
assert len(scaler.mean_) == len(selected_features), "Assertion 6 Failed: Scaler dimension mismatch!"
print(f"✓ Check 6 Passed: Validation and test sets transformed strictly via training-fitted scaler ({len(selected_features)} features).")
print("=" * 65)

# Summary Table of Scaled Feature Moments
moments_df = pd.DataFrame({
    "Train Mean": train_means,
    "Train Std": train_stds,
    "Val Mean": val_scaled_df[selected_features].mean(),
    "Val Std": val_scaled_df[selected_features].std()
})
print("\nSummary of Feature Moments across Train and Validation Sets:")
display(moments_df.round(4))

STAGE 5: SYSTEMATIC PIPELINE VALIDATION
✓ Check 1 Passed: Training and validation engine IDs do not overlap.
✓ Check 2 Passed: Engine counts are correct (Train: 80, Val: 20).
✓ Check 3 Passed: Training feature means are approx 0 (Max abs mean: 7.28e-12).
✓ Check 4 Passed: Training feature stds are approx 1 (Min: 1.0000, Max: 1.0000).
✓ Check 5 Passed: No missing or infinite values exist across all scaled matrices.
✓ Check 6 Passed: Validation and test sets transformed strictly via training-fitted scaler (17 features).

Summary of Feature Moments across Train and Validation Sets:


,Train Mean,Train Std,Val Mean,Val Std
op_setting_1,0.0,1.0,-0.0044,0.9924
op_setting_2,-0.0,1.0,0.0075,1.0074
sensor_2,-0.0,1.0,0.0339,1.0192
sensor_3,-0.0,1.0,0.0283,1.0374
sensor_4,-0.0,1.0,0.0309,1.0194
sensor_6,-0.0,1.0,-0.0267,1.0900
sensor_7,-0.0,1.0,-0.0308,1.0414
sensor_8,-0.0,1.0,0.0282,1.0239
sensor_9,0.0,1.0,-0.0350,0.8628
sensor_11,-0.0,1.0,0.0286,1.0254


### Summary of Preprocessing Pipeline
- **Engine-Wise Split:** Complete engine trajectories partitioned into **80 training engines (16,561 records)** and **20 validation engines (4,070 records)** with zero overlap.
- **Feature Pruning:** Out of 24 candidate features (3 settings + 21 sensors), **7 zero-variance features** (`op_setting_3`, `sensor_1`, `sensor_5`, `sensor_10`, `sensor_16`, `sensor_18`, `sensor_19`) were eliminated based on training set statistics.
- **Retained Features:** **17 features** (`op_setting_1`, `op_setting_2`, and 15 sensors including near-constant `sensor_6`).
- **Standardization:** `StandardScaler` fitted strictly on the 80 training engines, transforming train, validation, and test splits without data leakage.
- **Output Sets Ready for Windowing:**
  - `train_scaled_df`: `(16561, 20)` containing `unit`, `cycle`, 17 scaled features, and `RUL`.
  - `val_scaled_df`: `(4070, 20)` containing `unit`, `cycle`, 17 scaled features, and `RUL`.
  - `test_scaled_df`: `(13096, 19)` containing `unit`, `cycle`, and 17 scaled features.